# Nub Labs Playbook — Why Customers Buy Products Together

**Technique:** Association rule mining (Apriori algorithm)

This notebook has two parts:

**Part A — Curated dataset** (18 items, 267 transactions)
Reproduces the exact results shown on the interactive playbook page. Run this to verify every number on the site.

**Part B — UCI Online Retail Dataset** (541,909 rows, ~25,900 real transactions)
Runs the same algorithm on a real UK retailer's transaction history. Use this to explore at scale.

In [ ]:
!pip install mlxtend pandas openpyxl matplotlib -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')
print('Ready.')

---
## Part A — Curated Dataset

18 items, 267 transactions, same thresholds as the website.
Every rule shown on the playbook page was computed from this data.

In [ ]:
def repeat(items, n):
    return [items[:] for _ in range(n)]

transactions = (
    # Morning dairy shoppers
    repeat(['whole-milk', 'butter', 'yogurt'], 12) +
    repeat(['whole-milk', 'yogurt', 'tropical-fruit'], 10) +
    repeat(['whole-milk', 'butter', 'eggs'], 8) +

    # Bread & breakfast
    repeat(['bread', 'butter', 'coffee'], 12) +
    repeat(['bread', 'eggs', 'whole-milk'], 10) +
    repeat(['bread', 'coffee', 'eggs'], 6) +

    # Italian dinner basket
    repeat(['pasta', 'vegetables', 'red-meat'], 15) +
    repeat(['pasta', 'cheese', 'vegetables'], 10) +

    # Wine & entertaining
    repeat(['cheese', 'wine', 'red-meat'], 12) +
    repeat(['cheese', 'wine', 'tropical-fruit'], 8) +

    # Mixed staples
    repeat(['yogurt', 'tropical-fruit', 'bread'], 8) +
    repeat(['vegetables', 'tropical-fruit'], 5) +
    repeat(['coffee', 'whole-milk'], 4) +

    # Asian dinner basket
    repeat(['rice', 'chicken', 'vegetables'], 15) +
    repeat(['rice', 'chicken', 'herbs-spices'], 15) +
    repeat(['herbs-spices', 'vegetables', 'chicken'], 10) +

    # BBQ / casual entertaining
    repeat(['beer', 'red-meat', 'soda'], 12) +
    repeat(['beer', 'bread', 'soda'], 10) +

    # Sweet treats
    repeat(['chocolate', 'whole-milk'], 12) +
    repeat(['chocolate', 'coffee'], 12) +

    # Wine & fine dining
    repeat(['wine', 'cheese', 'herbs-spices'], 12) +
    repeat(['pasta', 'wine', 'cheese'], 8) +

    # Snacks & kids basket
    repeat(['soda', 'bread', 'chocolate'], 10) +

    # Budget staples
    repeat(['rice', 'eggs', 'bread'], 8) +

    # Coffee lovers
    repeat(['coffee', 'chocolate', 'whole-milk'], 8) +

    # Health basket
    repeat(['yogurt', 'tropical-fruit', 'whole-milk'], 7) +
    repeat(['chicken', 'vegetables', 'eggs'], 8)
)

print(f'Transactions: {len(transactions)}')
print(f'Unique items: {len(set(item for t in transactions for item in t))}')
print(f'Avg basket size: {np.mean([len(t) for t in transactions]):.2f} items')

In [ ]:
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_curated = pd.DataFrame(te_array, columns=te.columns_)

print('Item frequencies (support):')
freq = df_curated.mean().sort_values(ascending=False)
for item, sup in freq.items():
    print(f'  {item:<20} {sup:.3f} ({int(sup * len(transactions))} baskets)')

In [ ]:
frequent_curated = apriori(df_curated, min_support=0.07, use_colnames=True, max_len=3)
rules_curated = association_rules(frequent_curated, metric='lift', min_threshold=1.0)
rules_curated = rules_curated.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Frequent itemsets: {len(frequent_curated)}')
print(f'Association rules: {len(rules_curated)}')

In [ ]:
print('Top 20 rules by lift (matches website playbook):')
print()

top = rules_curated.head(20).copy()
top['antecedents'] = top['antecedents'].apply(lambda x: ', '.join(sorted(x)))
top['consequents'] = top['consequents'].apply(lambda x: ', '.join(sorted(x)))

for _, row in top.iterrows():
    print(f"  {{{row['antecedents']}}} → {{{row['consequents']}}}")
    print(f"  support={row['support']:.3f}  confidence={row['confidence']:.2%}  lift={row['lift']:.2f}x")
    print()

In [ ]:
# Verify the three key findings from the playbook's "What Surprised Us" section

def find_rule(rules_df, ant_set, con_set):
    for _, row in rules_df.iterrows():
        if set(row['antecedents']) == set(ant_set) and set(row['consequents']) == set(con_set):
            return row
    return None

findings = [
    (['beer'],  ['soda'],   'Beer → Soda (100% conf, strongest 2-item rule)'),
    (['wine'],  ['cheese'], 'Wine → Cheese (100% conf)'),
    (['rice'],  ['chicken'],'Rice → Chicken (strongest bidirectional cluster)'),
]

print('Verifying "What Surprised Us" claims from the playbook:')
print()
for ant, con, label in findings:
    r = find_rule(rules_curated, ant, con)
    if r is not None:
        print(f'  ✓ {label}')
        print(f'    conf={r["confidence"]:.2%}  lift={r["lift"]:.2f}x  support={r["support"]:.3f}')
    else:
        print(f'  ✗ {label} — rule not found at current thresholds')
    print()

# Whole milk popularity vs association strength
milk_rules = rules_curated[
    rules_curated['antecedents'].apply(lambda x: x == frozenset(['whole-milk']))
]
if not milk_rules.empty:
    max_lift = milk_rules['lift'].max()
    support = df_curated['whole-milk'].mean()
    print(f'  ✓ Whole milk popularity vs association strength')
    print(f'    support={support:.3f} ({support:.1%} of baskets) — most popular item')
    print(f'    max lift in any whole-milk rule: {max_lift:.2f}x — weaker than beer/wine/rice')

---
## Part B — UCI Online Retail Dataset

541,909 rows of real UK online retailer transactions (2010–2011).
Source: https://archive.ics.uci.edu/dataset/352/online+retail

**Option A (recommended):** Download the file from UCI, upload `Online Retail.xlsx` via the Colab Files panel, then run the cell with Option A uncommented.

**Option B:** Attempt a direct download from UCI. May fail due to rate limiting.

In [ ]:
df_retail = None

# ── Option A: Upload manually ─────────────────────────────────────────────────
# 1. Download Online Retail.xlsx from archive.ics.uci.edu/dataset/352/online+retail
# 2. In Colab: click the Files icon (left sidebar) → Upload
# 3. Uncomment the line below and run
#
# df_retail = pd.read_excel('Online Retail.xlsx')

# ── Option B: Direct download ─────────────────────────────────────────────────
import urllib.request
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
try:
    print('Downloading from UCI...')
    urllib.request.urlretrieve(url, 'Online Retail.xlsx')
    df_retail = pd.read_excel('Online Retail.xlsx')
    print(f'Loaded. Shape: {df_retail.shape}')
except Exception as e:
    print(f'Direct download failed ({e}).')
    print('Use Option A: upload the file manually.')

if df_retail is not None:
    print(f'\nColumns: {list(df_retail.columns)}')
    print(df_retail.head(3))

In [ ]:
if df_retail is not None:
    n_raw = len(df_retail)

    # Remove cancellations
    df_retail = df_retail[~df_retail['InvoiceNo'].astype(str).str.startswith('C')]
    # Remove missing CustomerID / Description
    df_retail = df_retail.dropna(subset=['CustomerID', 'Description'])
    # Remove returns (negative quantities)
    df_retail = df_retail[df_retail['Quantity'] > 0]
    # Clean description
    df_retail['Description'] = df_retail['Description'].str.strip().str.upper()
    # Remove non-product entries
    for kw in ['POSTAGE', 'MANUAL', 'BANK CHARGES', 'AMAZON FEE', 'CRUK']:
        df_retail = df_retail[~df_retail['Description'].str.contains(kw, na=False)]

    print(f'Raw rows:       {n_raw:,}')
    print(f'After cleaning: {len(df_retail):,}')
    print(f'Unique invoices:{df_retail["InvoiceNo"].nunique():,}')
    print(f'Unique products:{df_retail["Description"].nunique():,}')

In [ ]:
if df_retail is not None:
    basket_retail = df_retail.groupby('InvoiceNo')['Description'].apply(list).tolist()

    print(f'Transactions: {len(basket_retail):,}')
    print(f'Avg basket size: {np.mean([len(b) for b in basket_retail]):.1f} items')
    print(f'Max basket size: {max(len(b) for b in basket_retail)} items')

In [ ]:
if df_retail is not None:
    te_retail = TransactionEncoder()
    te_array_retail = te_retail.fit_transform(basket_retail)
    df_encoded_retail = pd.DataFrame(te_array_retail, columns=te_retail.columns_)

    print(f'Encoded shape: {df_encoded_retail.shape}')
    print(f'(rows = transactions, columns = unique products)')
    print(f'\nTop 10 most frequent products:')
    print(df_encoded_retail.mean().sort_values(ascending=False).head(10).to_string())

In [ ]:
if df_retail is not None:
    print('Finding frequent itemsets (min_support=0.02)...')
    frequent_retail = apriori(
        df_encoded_retail,
        min_support=0.02,
        use_colnames=True,
        max_len=3
    )
    frequent_retail['length'] = frequent_retail['itemsets'].apply(len)

    rules_retail = association_rules(frequent_retail, metric='lift', min_threshold=1.0)
    rules_retail = rules_retail.sort_values('lift', ascending=False).reset_index(drop=True)

    print(f'Frequent itemsets: {len(frequent_retail):,}')
    print(f'  1-item: {len(frequent_retail[frequent_retail.length==1]):,}')
    print(f'  2-item: {len(frequent_retail[frequent_retail.length==2]):,}')
    print(f'  3-item: {len(frequent_retail[frequent_retail.length==3]):,}')
    print(f'Total rules: {len(rules_retail):,}')

In [ ]:
if df_retail is not None:
    print('Top 20 rules by lift:')
    print()
    top_retail = rules_retail.head(20).copy()
    top_retail['antecedents'] = top_retail['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    top_retail['consequents'] = top_retail['consequents'].apply(lambda x: ', '.join(sorted(x)))
    top_retail['support'] = top_retail['support'].map('{:.4f}'.format)
    top_retail['confidence'] = top_retail['confidence'].map('{:.2%}'.format)
    top_retail['lift'] = top_retail['lift'].map('{:.2f}x'.format)
    print(top_retail[['antecedents','consequents','support','confidence','lift']].to_string(index=False))

In [ ]:
if df_retail is not None:
    fig, ax = plt.subplots(figsize=(10, 7))
    scatter = ax.scatter(
        rules_retail['support'],
        rules_retail['confidence'],
        c=rules_retail['lift'],
        cmap='YlOrRd',
        alpha=0.5,
        s=25,
        edgecolors='none'
    )
    plt.colorbar(scatter, label='Lift')
    ax.set_xlabel('Support')
    ax.set_ylabel('Confidence')
    ax.set_title('UCI Online Retail: Association Rules (support vs confidence, coloured by lift)')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Key insight
    print('Key observation from the UCI data:')
    top_lift_rules = rules_retail[rules_retail['lift'] > 5]
    print(f'  Rules with lift > 5x: {len(top_lift_rules)}')
    print(f'  These tend to have LOW support — niche but strong associations.')
    print(f'  Mirrors the curated dataset finding: popularity does not equal association strength.')

---
## Next Steps

- **Adjust thresholds:** Try `min_support=0.01` for more rules, or `min_support=0.05` for fewer but stronger ones.
- **Filter by lift:** `rules[rules.lift > 3]` shows only strong associations.
- **Try FP-Growth:** `from mlxtend.frequent_patterns import fpgrowth` — same interface, faster on large datasets.
- **Apply to your data:** Replace the `transactions` list in Part A with your own data. The rest of the code is identical.